## Finetuning a Language Model with LoRA

This notebook demonstrates how to fine-tune a pre-trained language model using LoRA (Low-Rank Adaptation) for text generation. We will use the `bigscience/bloomz-560m` model and the `Abirate/english_quotes` dataset.

### 1. Install necessary libraries

In [1]:
# This cell is now empty as installation will be handled in the next cell.
# The previous installation of peft==0.5.0 was still causing compatibility issues.

In [2]:
# Create cache directory
!mkdir -p cache

# Uninstall existing versions of potentially conflicting libraries to ensure a clean state
# The -y flag confirms the uninstallation without prompting
!pip uninstall -y peft transformers accelerate datasets

# Install the latest compatible versions of all required libraries
# This is the most reliable way to resolve persistent dependency conflicts.
!pip install datasets peft transformers accelerate


Found existing installation: peft 0.5.0
Uninstalling peft-0.5.0:
  Successfully uninstalled peft-0.5.0
Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfu

### 2. Load a pre-trained language model and its tokenizer

In [3]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Define the model name
model_name = "bigscience/bloomz-560m"

# Load the tokenizer and the foundation model
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad_token_id for the tokenizer if it's not already set
# This is important for batching and generation with some models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    foundation_model.config.pad_token_id = foundation_model.config.eos_token_id

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

### 3. Load the dataset and preprocess it for the model

In [4]:
# Load the 'Abirate/english_quotes' dataset
data = load_dataset("Abirate/english_quotes")

# Sample 10% of the training split
# The dataset has only one split named 'train'
train_size = len(data["train"])
sample_size = int(0.1 * train_size)
sampled_data = data["train"].shuffle(seed=42).select(range(sample_size))

# Tokenize the dataset
def tokenize_function(examples):
    # The tokenizer expects input as 'text' or similar, here we use 'quote'
    return tokenizer(examples["quote"], truncation=True, max_length=128)

tokenized_data = sampled_data.map(tokenize_function, batched=True, remove_columns=["quote", "author", "tags"])

# Prepare for causal language modeling by setting labels
tokenized_data = tokenized_data.map(lambda examples: {'labels': examples['input_ids']}, batched=True)

# Display a sample of the preprocessed data
display(tokenized_data.to_pandas().head())

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

,input_ids,attention_mask,labels
0,"[123916, 5926, 19142, 16997, 21445, 262, 15, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[123916, 5926, 19142, 16997, 21445, 262, 15, 1..."
1,"[181046, 29998, 1306, 1130, 7086, 427, 722, 60...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[181046, 29998, 1306, 1130, 7086, 427, 722, 60..."
2,"[187034, 78831, 9631, 388, 4345, 361, 20886, 7...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[187034, 78831, 9631, 388, 4345, 361, 20886, 7..."
3,"[119533, 137656, 1728, 15, 722, 368, 33136, 98...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[119533, 137656, 1728, 15, 722, 368, 33136, 98..."
4,"[123916, 1400, 19481, 16916, 16612, 149160, 52...","[1, 1, 1, 1, 1, 1, 1, 1, 1]","[123916, 1400, 19481, 16916, 16612, 149160, 52..."


### 4. Configure LoRA using `LoraConfig` and apply it to the pre-trained model

In [5]:
from peft import LoraConfig, get_peft_model

# Define LoRA configuration
lora_config = LoraConfig(
    r=16, # LoRA attention dimension
    lora_alpha=16, # Alpha parameter for LoRA scaling
    target_modules=["query_key_value"], # Target modules for LoRA adaptation, specific to BLOOM models
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Do not train bias terms
    task_type="CAUSAL_LM" # Task type for causal language modeling
)

# Add the adapter layers to the foundation model
peft_model = get_peft_model(foundation_model, lora_config)

# Print the number of trainable parameters
print("Trainable parameters of the PEFT model:")
print(peft_model.print_trainable_parameters())

Trainable parameters of the PEFT model:
trainable params: 1,572,864 || all params: 560,787,456 || trainable%: 0.2804741766549072
None


### 5. Configure the training arguments

In [12]:
import transformers
from transformers import TrainingArguments
import os

# Define output directory for training artifacts
output_directory = os.path.join("../cache/working", "peft_lab_outputs")

# Configure training arguments
training_args = TrainingArguments(
    report_to="none", # Disable reporting to external services
    output_dir=output_directory,
    auto_find_batch_size=True, # Automatically find a suitable batch size
    learning_rate=3e-2, # Higher learning rate than full fine-tuning.
    num_train_epochs=3, # Number of training epochs
    per_device_train_batch_size=4, # Define a batch size if auto_find_batch_size is not effective
    gradient_accumulation_steps=4, # Accumulate gradients over multiple steps
    evaluation_strategy="no", # No evaluation during training
    save_strategy="epoch", # Save model at the end of each epoch
    logging_dir=f"{output_directory}/logs",
    logging_strategy="steps",
    logging_steps=50,
    use_cpu=True # Use CPU for training (for demonstration purposes, GPU is usually preferred)
)

ImportError: cannot import name 'PeftMixedModel' from 'peft' (/usr/local/lib/python3.12/dist-packages/peft/__init__.py)

### 6. Initialize and train the model using `Trainer`

In [7]:
from transformers import Trainer, DataCollatorForLanguageModeling

# Initialize the Trainer
trainer = Trainer(
    model=peft_model, # The PEFT-adapted model
    args=training_args, # Training arguments
    train_dataset=tokenized_data, # The tokenized and preprocessed training dataset
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False) # Data collator for causal language modeling
)

# Start training
print("Starting training...")
trainer.train()
print("Training finished.")

ImportError: cannot import name 'PeftMixedModel' from 'peft' (/usr/local/lib/python3.12/dist-packages/peft/__init__.py)

### 7. Save the optimized LoRA model

In [13]:
import time
import os

# Generate a timestamp for the model path
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

# Save the fine-tuned PEFT model
trainer.model.save_pretrained(peft_model_path)
print(f"LoRA model saved to: {peft_model_path}")

NameError: name 'output_directory' is not defined

### 8. Load the saved LoRA model for inference

In [9]:
from peft import PeftModel

# Load the base model again for fresh inference (optional, can use peft_model directly)
# It's good practice to load the base model and then attach the adapter for inference
# foundation_model_inference = AutoModelForCausalLM.from_pretrained(model_name)

# Load the PEFT model from the saved path onto the foundation model
# If you run this cell in a new session, ensure 'foundation_model' is loaded first
inference_model = PeftModel.from_pretrained(foundation_model, peft_model_path, is_trainable=False)
print("LoRA model loaded for inference.")

NameError: name 'peft_model_path' is not defined

### 9. Generate text using the fine-tuned model

In [10]:
import torch

# Input text for generation
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

# Ensure the model is in evaluation mode
inference_model.eval()

# Generate output tokens
with torch.no_grad(): # Disable gradient calculation for inference
    outputs = inference_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=50, # Maximum number of new tokens to generate
        num_beams=5, # Use beam search for better quality generation
        do_sample=True, # Enable sampling
        top_k=50, # Sample from top_k most likely next words
        top_p=0.95, # Sample from the smallest set of words whose cumulative probability exceeds top_p
        temperature=0.7, # Control randomness of predictions
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# Decode and print the generated text
print("Generated Text:")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

NameError: name 'inference_model' is not defined